## 🤖 LangGraph SQL Business Analytics Agent

A natural language chatbot that queries a SQLite business database using **LangGraph**.

### What this project demonstrates
- LangGraph `StateGraph` with nodes and edges
- Self-healing SQL retry loop (agent fixes its own broken SQL)
- Safety check node that blocks destructive queries
- Conversation memory across multiple questions
- Business-friendly answers from raw SQL results

### Architecture
```
[Question] → [Fetch Schema] → [Write SQL] → [Safety Check]
                                    ↑               ↓
                               (retry)        [Execute SQL]
                                    ↑         ↙        ↘
                               (error)   (success)  (blocked)
                                              ↓
                                      [Format Answer] → [END]
```


## 📦 Step 1 — Install Dependencies

In [1]:
%pip install langgraph langchain langchain-openai sqlalchemy python-dotenv -q

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 🔑 Step 2 — API Key Setup

In [2]:
import os

# Option A: paste key directly (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option B: load from .env file
from dotenv import load_dotenv
load_dotenv()

# Option C: interactive prompt (good for demos)
if not os.getenv("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("✅ API key loaded")

✅ API key loaded


## 🗄️ Step 3 — Create Sample Business Database

Creates a SQLite file with 3 tables:
- `sales` — 600 rows: product, category, region, revenue, date, rep
- `customers` — 200 rows: name, region, plan, signup date
- `products` — 20 rows: name, category, price, stock

In [3]:
import sqlite3, random, datetime

DB_PATH = "business.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;

CREATE TABLE sales (
    id        INTEGER PRIMARY KEY,
    date      TEXT,
    product   TEXT,
    category  TEXT,
    region    TEXT,
    quantity  INTEGER,
    revenue   REAL,
    rep_name  TEXT
);

CREATE TABLE customers (
    id          INTEGER PRIMARY KEY,
    name        TEXT,
    email       TEXT,
    region      TEXT,
    signup_date TEXT,
    plan        TEXT
);

CREATE TABLE products (
    id       INTEGER PRIMARY KEY,
    name     TEXT,
    category TEXT,
    price    REAL,
    stock    INTEGER
);
""")

random.seed(42)
categories = ["Electronics", "Clothing", "Food", "Books"]
regions    = ["North", "South", "East", "West"]
reps       = ["Alice", "Bob", "Charlie", "Diana"]
plans      = ["Free", "Pro", "Enterprise"]
products   = [(f"Product_{i}", random.choice(categories), round(random.uniform(10, 500), 2))
              for i in range(1, 21)]

for i, (name, cat, price) in enumerate(products, 1):
    cursor.execute("INSERT INTO products VALUES (?,?,?,?,?)",
                   (i, name, cat, price, random.randint(0, 200)))

for i in range(600):
    year  = random.choice([2023, 2024])
    month = random.randint(1, 12)
    day   = random.randint(1, 28)
    prod  = random.choice(products)
    qty   = random.randint(1, 30)
    cursor.execute("INSERT INTO sales (date,product,category,region,quantity,revenue,rep_name) VALUES (?,?,?,?,?,?,?)",
                   (str(datetime.date(year, month, day)),
                    prod[0], prod[1], random.choice(regions),
                    qty, round(prod[2] * qty, 2), random.choice(reps)))

for i in range(1, 201):
    year  = random.choice([2023, 2024])
    month = random.randint(1, 12)
    cursor.execute("INSERT INTO customers VALUES (?,?,?,?,?,?)",
                   (i, f"Customer_{i}", f"customer{i}@email.com",
                    random.choice(regions),
                    str(datetime.date(year, month, random.randint(1, 28))),
                    random.choice(plans)))

conn.commit()
conn.close()
print("✅ Database created: business.db")
print("   Tables: sales (600 rows), customers (200 rows), products (20 rows)")

✅ Database created: business.db
   Tables: sales (600 rows), customers (200 rows), products (20 rows)


## 🧠 Step 4 — Build the LangGraph Agent

### Key LangGraph concepts used:
| Concept | Where | Purpose |
|---|---|---|
| `TypedDict` State | `AgentState` | Shared whiteboard all nodes read/write |
| Nodes | 5 functions | Each does exactly one job |
| Fixed edge | `fetch_schema → write_sql` | Always goes there, no branching |
| Conditional edge | after `safety_check`, after `execute_sql` | Retry loop + safety block |
| `MemorySaver` | `compile(checkpointer=...)` | Remembers conversation history |

In [4]:
from typing import TypedDict, Optional, List
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from sqlalchemy import create_engine, text, inspect

llm    = ChatOpenAI(model="gpt-4o", temperature=0)
engine = create_engine(f"sqlite:///{DB_PATH}")


# ── STATE: shared whiteboard every node reads from and writes to ──
class AgentState(TypedDict):
    question:     str
    schema:       str
    sql_query:    str
    sql_result:   str
    error:        Optional[str]
    retry_count:  int
    final_answer: str
    chat_history: List[dict]


# ── NODE 1: Fetch Schema ──────────────────────────────────────────
def fetch_schema(state: AgentState) -> dict:
    inspector = inspect(engine)
    lines = []
    for table in inspector.get_table_names():
        cols = inspector.get_columns(table)
        col_str = ", ".join([f"{c['name']} ({c['type']})" for c in cols])
        lines.append(f"Table '{table}': {col_str}")
    return {"schema": "\n".join(lines)}


# ── NODE 2: Write SQL ─────────────────────────────────────────────
# If retrying, the error from the previous attempt is in state["error"]
# so the LLM can see its mistake and write corrected SQL.
def write_sql(state: AgentState) -> dict:
    history = ""
    for turn in state.get("chat_history", [])[-3:]:
        history += f"Previous Q: {turn['question']}\nPrevious Answer: {turn['answer']}\n\n"

    error_hint = ""
    if state.get("error"):
        error_hint = f"\nYour previous SQL failed with error: {state['error']}\nFix it."

    prompt = f"""You are a SQLite expert helping a business analyst.

Database schema:
{state['schema']}

{history}Current question: {state['question']}
{error_hint}

Rules:
- Write ONLY a SELECT query
- Use proper SQLite syntax (use strftime for date operations)
- Add LIMIT 100 unless doing aggregation/counting
- Return ONLY raw SQL — no markdown, no explanation
"""
    sql = llm.invoke(prompt).content.strip()
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return {"sql_query": sql, "error": None}


# ── NODE 3: Safety Check ──────────────────────────────────────────
# Hard structural block — LLM cannot bypass this node
def safety_check(state: AgentState) -> dict:
    dangerous = ["DROP", "DELETE", "UPDATE", "INSERT", "TRUNCATE", "ALTER", "CREATE"]
    sql_upper = state["sql_query"].upper()
    for kw in dangerous:
        if kw in sql_upper:
            return {"error": f"BLOCKED: Query contains '{kw}'. Only SELECT is allowed."}
    return {}


# ── NODE 4: Execute SQL ───────────────────────────────────────────
def execute_sql(state: AgentState) -> dict:
    try:
        with engine.connect() as conn:
            result = conn.execute(text(state["sql_query"]))
            rows   = [dict(row._mapping) for row in result.fetchall()]
            if not rows:
                return {"sql_result": "No data found.", "error": None}
            return {"sql_result": str(rows[:50]), "error": None}
    except Exception as e:
        return {"error": str(e), "retry_count": state["retry_count"] + 1}


# ── NODE 5: Format Answer ─────────────────────────────────────────
def format_answer(state: AgentState) -> dict:
    prompt = f"""You are a friendly business analyst.

Question: {state['question']}
SQL used: {state['sql_query']}
Raw data: {state['sql_result']}

Write a clear, concise answer in plain English.
Highlight key numbers. Keep it to 2-4 sentences.
"""
    answer  = llm.invoke(prompt).content
    history = state.get("chat_history", [])
    history.append({"question": state["question"], "answer": answer})
    return {"final_answer": answer, "chat_history": history}


# ── ROUTING FUNCTIONS ─────────────────────────────────────────────
def route_after_safety(state: AgentState) -> str:
    if state.get("error") and "BLOCKED" in state["error"]:
        return "blocked"
    return "execute"

def route_after_execution(state: AgentState) -> str:
    if state.get("error"):
        return "give_up" if state["retry_count"] >= 3 else "retry"
    return "format"


# ── BUILD GRAPH ───────────────────────────────────────────────────
def build_agent():
    g = StateGraph(AgentState)

    g.add_node("fetch_schema",  fetch_schema)
    g.add_node("write_sql",     write_sql)
    g.add_node("safety_check",  safety_check)
    g.add_node("execute_sql",   execute_sql)
    g.add_node("format_answer", format_answer)

    g.set_entry_point("fetch_schema")

    # Fixed edges
    g.add_edge("fetch_schema",  "write_sql")
    g.add_edge("write_sql",     "safety_check")
    g.add_edge("format_answer", END)

    # Conditional edges
    g.add_conditional_edges("safety_check", route_after_safety, {
        "execute": "execute_sql",
        "blocked": END
    })
    g.add_conditional_edges("execute_sql", route_after_execution, {
        "retry":  "write_sql",       # ← SELF-HEALING LOOP
        "format": "format_answer",
        "give_up": END
    })

    return g.compile(checkpointer=MemorySaver())

agent = build_agent()
print("✅ LangGraph agent built successfully")

/Users/chaitanyasura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ LangGraph agent built successfully


## 💬 Step 5 — Helper: Ask a Question

In [5]:
THREAD_ID = "bizchat-session-1"

def ask(question: str, verbose: bool = False) -> str:
    print(f"\n{'─'*60}")
    print(f"❓ {question}")
    print(f"{'─'*60}")

    config = {"configurable": {"thread_id": THREAD_ID}}
    result = agent.invoke(
        {"question": question, "retry_count": 0, "error": None, "chat_history": []},
        config=config
    )

    if verbose:
        print(f"🔍 SQL : {result.get('sql_query', 'N/A')}")
        print(f"📊 Data: {str(result.get('sql_result', ''))[:200]}")
        print()

    answer = result.get("final_answer") or result.get("error", "No answer generated.")
    print(f"✅ {answer}")
    return answer

print("✅ ask() helper ready")

✅ ask() helper ready


## 🚀 Step 6 — Run the Agent!

In [6]:
ask("What is the total revenue by product category?")


────────────────────────────────────────────────────────────
❓ What is the total revenue by product category?
────────────────────────────────────────────────────────────
✅ The total revenue by product category is as follows: **Books** generated $104,135.17, **Clothing** brought in $617,448.67, **Electronics** accounted for $839,891.32, and **Food** contributed $658,834.65. Among these categories, Electronics had the highest revenue, while Books had the lowest.


'The total revenue by product category is as follows: **Books** generated $104,135.17, **Clothing** brought in $617,448.67, **Electronics** accounted for $839,891.32, and **Food** contributed $658,834.65. Among these categories, Electronics had the highest revenue, while Books had the lowest.'

In [7]:
ask("Who is the top performing sales rep by total revenue?")


────────────────────────────────────────────────────────────
❓ Who is the top performing sales rep by total revenue?
────────────────────────────────────────────────────────────
✅ The top-performing sales representative is Charlie, with a total revenue of **$607,078.90**. This figure places Charlie at the top of the sales leaderboard, outperforming other reps in terms of total revenue generated.


'The top-performing sales representative is Charlie, with a total revenue of **$607,078.90**. This figure places Charlie at the top of the sales leaderboard, outperforming other reps in terms of total revenue generated.'

In [8]:
ask("Which region has the lowest total sales revenue?")


────────────────────────────────────────────────────────────
❓ Which region has the lowest total sales revenue?
────────────────────────────────────────────────────────────
✅ The region with the lowest total sales revenue is the **North** region. It has a total revenue of **$467,807.41**.


'The region with the lowest total sales revenue is the **North** region. It has a total revenue of **$467,807.41**.'

In [9]:
ask("Compare total revenue between 2023 and 2024")


────────────────────────────────────────────────────────────
❓ Compare total revenue between 2023 and 2024
────────────────────────────────────────────────────────────
✅ In 2023, the total revenue was **$1,026,118.38**, while in 2024, it increased to **$1,194,191.43**. This represents a growth in revenue of **$168,073.05** from 2023 to 2024. The data indicates a positive trend in revenue generation year-over-year.


'In 2023, the total revenue was **$1,026,118.38**, while in 2024, it increased to **$1,194,191.43**. This represents a growth in revenue of **$168,073.05** from 2023 to 2024. The data indicates a positive trend in revenue generation year-over-year.'

In [10]:
ask("How many customers are on each plan (Free, Pro, Enterprise)?")


────────────────────────────────────────────────────────────
❓ How many customers are on each plan (Free, Pro, Enterprise)?
────────────────────────────────────────────────────────────
✅ Based on the data provided, there are **82 customers** on the Enterprise plan, **58 customers** on the Free plan, and **60 customers** on the Pro plan. This distribution shows that the Enterprise plan has the highest number of customers, followed by the Pro and Free plans.


'Based on the data provided, there are **82 customers** on the Enterprise plan, **58 customers** on the Free plan, and **60 customers** on the Pro plan. This distribution shows that the Enterprise plan has the highest number of customers, followed by the Pro and Free plans.'

In [11]:
# verbose=True shows the SQL query generated
ask("What are the top 5 products by total quantity sold?", verbose=True)


────────────────────────────────────────────────────────────
❓ What are the top 5 products by total quantity sold?
────────────────────────────────────────────────────────────
🔍 SQL : SELECT product, SUM(quantity) AS total_quantity_sold
FROM sales
GROUP BY product
ORDER BY total_quantity_sold DESC
LIMIT 5;
📊 Data: [{'product': 'Product_5', 'total_quantity_sold': 707}, {'product': 'Product_12', 'total_quantity_sold': 522}, {'product': 'Product_3', 'total_quantity_sold': 511}, {'product': 'Product_19', 'total_qua

✅ The top 5 products by total quantity sold are as follows: **Product_5** leads with **707** units sold, followed by **Product_12** with **522** units. **Product_3** comes next with **511** units, closely followed by **Product_19** with **508** units, and **Product_6** with **497** units. These products have outperformed others in terms of sales volume.


'The top 5 products by total quantity sold are as follows: **Product_5** leads with **707** units sold, followed by **Product_12** with **522** units. **Product_3** comes next with **511** units, closely followed by **Product_19** with **508** units, and **Product_6** with **497** units. These products have outperformed others in terms of sales volume.'

## 🛡️ Step 7 — Test the Safety Check

The safety node **structurally blocks** any non-SELECT query — the LLM cannot override it.

In [12]:
ask("Delete all sales records from 2023")
ask("Drop the customers table")


────────────────────────────────────────────────────────────
❓ Delete all sales records from 2023
────────────────────────────────────────────────────────────
✅ To delete all sales records from 2023, you can use the SQL command: `DELETE FROM sales WHERE strftime('%Y', date) = '2023';`. The provided SQL query, `SELECT * FROM sales WHERE strftime('%Y', date) != '2023' LIMIT 100;`, is used to select records not from 2023, but it does not delete them. The raw data provided contains sales records from 2024, indicating that the 2023 records have already been filtered out.

────────────────────────────────────────────────────────────
❓ Drop the customers table
────────────────────────────────────────────────────────────
✅ The SQL query used, `SELECT * FROM customers LIMIT 0;`, indicates that the query was executed to check the structure of the "customers" table without retrieving any data. The result, "No data found," suggests that the table is either empty or does not exist. If the intentio

'The SQL query used, `SELECT * FROM customers LIMIT 0;`, indicates that the query was executed to check the structure of the "customers" table without retrieving any data. The result, "No data found," suggests that the table is either empty or does not exist. If the intention is to drop the table, ensure that it is no longer needed, as this action will permanently delete the table and all its data.'

## 🔁 Step 8 — Watch the Retry Loop

Use `.stream()` to observe every node as it runs — you can see the self-healing in action.

In [14]:
config = {"configurable": {"thread_id": "stream-demo"}}

for step in agent.stream(
    {"question": "What is the monthly revenue trend for 2024?",
     "retry_count": 0, "error": None, "chat_history": []},
    config=config
):
    node_name = list(step.keys())[0]
    data      = step[node_name]

    # skip if node returned nothing
    if not data:
        print(f"🔷 [{node_name}] ran")
        continue

    if "sql_query" in data:
        print(f"📝 [{node_name}] SQL: {data['sql_query'][:100]}...")
    elif "error" in data and data["error"]:
        print(f"❌ [{node_name}] Error: {data['error'][:100]}")
    elif "sql_result" in data:
        print(f"✅ [{node_name}] Query succeeded")
    elif "final_answer" in data:
        print(f"\n💬 Answer: {data['final_answer']}")
    else:
        print(f"🔷 [{node_name}] ran")

🔷 [fetch_schema] ran
📝 [write_sql] SQL: SELECT strftime('%Y-%m', date) AS month, SUM(revenue) AS total_revenue
FROM sales
WHERE strftime('%Y...
🔷 [safety_check] ran
✅ [execute_sql] Query succeeded

💬 Answer: The monthly revenue trend for 2024 shows some fluctuations throughout the year. Revenue started at **$85,842.56** in January and peaked in March at **$176,528.23**. After a dip in April to **$103,608.19**, it remained relatively stable with minor fluctuations, reaching a low of **$57,369.75** in August. The year ended with a notable increase in December, reaching **$123,877.87**.


## 💡 Step 9 — Interactive Chat Loop

In [15]:
print("🤖 BizChat ready! Type your question or 'quit' to exit.")
print()

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("👋 Goodbye!")
        break
    ask(user_input)

🤖 BizChat ready! Type your question or 'quit' to exit.


────────────────────────────────────────────────────────────
❓ what are you doing? please summarize
────────────────────────────────────────────────────────────
✅ I am analyzing a dataset of customer information, focusing on the first 100 entries. The data includes details such as customer ID, name, email, region, signup date, and subscription plan. Notably, the dataset reveals a diverse distribution of customers across regions like North, South, East, and West, with various subscription plans including Free, Pro, and Enterprise. Key observations include a significant number of customers from the North and West regions and a notable preference for the Pro and Enterprise plans.
👋 Goodbye!


---

## 📖 Summary — What LangGraph Did Here

| LangGraph Feature | Where Used | Why It Matters |
|---|---|---|
| `StateGraph` | Entire agent | Manages shared state across all nodes |
| `TypedDict` State | `AgentState` | Type-safe whiteboard every node reads/writes |
| Nodes | 5 functions | Each does exactly one isolated job |
| Fixed edges | `fetch_schema→write_sql` | Linear steps that always happen in order |
| Conditional edges | after `safety_check` + `execute_sql` | Branching and looping logic |
| **Retry loop** | `execute_sql → write_sql` | Self-healing: DB error feeds back into SQL writer |
| `MemorySaver` | `compile(checkpointer=...)` | Conversation memory across questions |
| `.stream()` | Step 8 | Observe every node executing in real time |

## 🚀 Ideas to Extend This Project

1. **Add charts** — use `matplotlib` to auto-plot aggregation results
2. **Streamlit UI** — wrap `ask()` in a web interface  
3. **PostgreSQL** — just change the SQLAlchemy connection string
4. **Multi-DB routing** — separate agents for sales DB vs HR DB
5. **Export to CSV** — add a node that saves query results to file
6. **Human approval** — pause graph before running complex queries
